# Let's go PRO!

Advanced RAG Techniques!

Let's start by digging into ingest:

1. No LangChain! Just native for maximum flexibility
2. Let's use an LLM to divide up chunks in a sensible way
3. Let's use the best chunk size and encoder from yesterday
4. Let's also have the LLM rewrite chunks in a way that's most useful ("document pre-processing")

In [22]:
print("skf")

skf


In [2]:
from pathlib import Path
from google import genai
import json
import os
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go


load_dotenv(override=True)

MODEL = "models/gemini-3.1-flash-lite"

DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "models/gemini-embedding-2"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

client1 = genai.Client(api_key = os.getenv("key1"))
# client2 = genai.Client(api_key = os.getenv("key2"))
# client3 = genai.Client(api_key = os.getenv("key3"))
# client4 = genai.Client(api_key = os.getenv("key4"))
# client5 = genai.Client(api_key = os.getenv("key5"))

# client = OpenAI(
#     api_key= os.getenv("GROQ"),
#     base_url="https://api.groq.com/openai/v1"
# )

In [3]:
for model in client1.models.list():
    if "generateContent" in model.supported_actions:
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

In [4]:
for m in client1.models.list():
    if "embedContent" in m.supported_actions:
        print(m.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [5]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [ ]:
# A class to perfectly represent a chunk


class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)

#list of chunks on chunk class
class Chunks(BaseModel):
    chunks: list[Chunk]

## Three steps:

1. Fetch documents from the knowledge base, like LangChain did
2. Call an LLM to turn documents into Chunks
3. Store the Chunks in Chroma

That's it!

### Let's start with Step 1

In [ ]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name #gives foldername as str
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

    # returns list of doc(dict type) 
    # [{"type": "books", "source": "knowledge_base/books/python.md", "text": "Python supports OOP."}, {}, {}....etc]


In [8]:
documents = fetch_documents()
# stores list of docs
# # [{"type": "books", "source": "knowledge_base/books/python.md", "text": "Python supports OOP."}, {}, {}....etc]

Loaded 76 documents


In [9]:
print(type(documents[0]))
print(type(documents))
print((documents[:2]))

<class 'dict'>
<class 'list'>
[{'type': 'company', 'source': 'knowledge-base/company/about.md', 'text': "# About Insurellm\n\nInsurellm was founded by Avery Lancaster in 2015 as an insurance tech startup designed to disrupt an industry in need of innovative products. Its first product was Markellm, the marketplace connecting consumers with insurance providers.\n\nThe company experienced rapid growth in its first five years, expanding its product portfolio to include Carllm (auto insurance portal), Homellm (home insurance portal), and Rellm (enterprise reinsurance platform). By 2020, Insurellm had reached a peak of 200 employees with 12 offices across the US.\n\nHowever, the company underwent a strategic restructuring in 2022-2023 to focus on profitability and sustainable growth. This included consolidating office locations, implementing a remote-first strategy, and streamlining operations. As of 2025, Insurellm operates with a lean, highly efficient team of 32 employees who have built 

### Donezo! On to Step 2 - make the chunks

In [10]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Return the result as valid JSON only.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [11]:
print(make_prompt(documents[0]))


You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: company
The document has been retrieved from: knowledge-base/company/about.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 5 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Return the result as valid JSON only.

Here is the document:

# About Insur

In [12]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [14]:
make_messages(documents[0])

[{'role': 'user',
  'content': "\nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm.\nThe document is of type: company\nThe document has been retrieved from: knowledge-base/company/about.md\n\nA chatbot will use these chunks to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.\nThis document should probably be split into 5 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk.\nTogether your chunks should represent the entire document with overlap.\n\nReturn the result as valid J

In [ ]:
#gemini call

# def process_document(document):
#     messages = make_messages(document)

#     response = client.models.generate_content(
#         model=MODEL,
#         contents=messages[-1]["content"],   # use the user prompt
#         config={
#             "response_mime_type": "application/json",
#             "response_schema": Chunks,
#         },
#     )

#     # If your SDK supports parsed output
#     if hasattr(response, "parsed") and response.parsed is not None:
#         doc_as_chunks = response.parsed.chunks
#     else:
#         # Fallback for older SDK versions
#         doc_as_chunks = Chunks.model_validate_json(response.text).chunks

#     return [chunk.as_result(document) for chunk in doc_as_chunks]

In [13]:
#chunking part

import time

LAST_CALL_TIME = 0

def process_document(document):
    global LAST_CALL_TIME

    while True:
        # it keeps under 15 RPM
        elapsed = time.time() - LAST_CALL_TIME
        if elapsed < 4:
            time.sleep(4 - elapsed)

        try:
            messages = make_messages(document)
            response = client1.models.generate_content(
                model=MODEL,
                contents=messages[-1]["content"],
                config={
                    "response_mime_type": "application/json",
                    "response_schema": Chunks,
                },
            )

            LAST_CALL_TIME = time.time()

            if hasattr(response, "parsed") and response.parsed is not None:
                doc_as_chunks = response.parsed.chunks
            else:
                doc_as_chunks = Chunks.model_validate_json(response.text).chunks

            return [chunk.as_result(document) for chunk in doc_as_chunks]

        except Exception as e:
            if "429" in str(e):
                print("Rate limit hit. Waiting 60 seconds...")
                time.sleep(60)
            else:
                raise

In [ ]:
# def process_document(document):
#     messages = make_messages(document)

    

#     response = client.chat.completions.create(
#         model=MODEL,
#         messages=messages,
#         response_format={"type": "json_object"},
#     )

#     reply = response.choices[0].message.content
#     doc_as_chunks = Chunks.model_validate_json(reply).chunks

#     return [chunk.as_result(document) for chunk in doc_as_chunks]

In [14]:
doc1 = process_document(documents[0])
print(type(doc1))

<class 'list'>


In [15]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc)) #[a, b] , [c, d, e] ====> [a, b, c, d, e]
    return chunks

In [16]:
chunks = create_chunks(documents)

100%|██████████| 76/76 [12:49<00:00, 10.12s/it]


In [17]:
print(len(chunks))

577


### Well that was easy! If a bit slow.

In the python module version, I sneakily use the multi-processing Pool to run this in parallel,
but if you get a Rate Limit Error you can turn this off in the code.

### Finally, Step 3 - save the embeddings

In [ ]:
# openai embedding

# def create_embeddings(chunks):
#     chroma = PersistentClient(path=DB_NAME)
#     if collection_name in [c.name for c in chroma.list_collections()]:
#         chroma.delete_collection(collection_name)

#     texts = [chunk.page_content for chunk in chunks]
#     emb = client1.embeddings.create(model=embedding_model, input=texts).data
#     vectors = [e.embedding for e in emb]

#     collection = chroma.get_or_create_collection(collection_name)

#     ids = [str(i) for i in range(len(chunks))]
#     metas = [chunk.metadata for chunk in chunks]

#     collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
#     print(f"Vectorstore created with {collection.count()} documents")

In [18]:
import time
from chromadb import PersistentClient

def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)

    # Delete old collection if it exists
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    collection = chroma.get_or_create_collection(collection_name)

    BATCH_SIZE = 100

    for start in range(0, len(chunks), BATCH_SIZE):

        batch = chunks[start:start + BATCH_SIZE]

        texts = [chunk.page_content for chunk in batch]
        ids = [str(i) for i in range(start, start + len(batch))]
        metas = [chunk.metadata for chunk in batch]

        while True:
            try:
                response = client1.models.embed_content(
                    model=embedding_model,
                    contents=texts
                )

                vectors = [emb.values for emb in response.embeddings]

                collection.add(
                    ids=ids,
                    embeddings=vectors,
                    documents=texts,
                    metadatas=metas,
                )

                print(
                    f"✅ Batch {(start // BATCH_SIZE) + 1} added "
                    f"({len(batch)} chunks)"
                )

                # Optional: small pause between successful batches
                time.sleep(2)

                break

            except Exception as e:
                if "429" in str(e):
                    print("⚠️ Rate limit reached. Waiting 25 seconds...")
                    time.sleep(35)
                else:
                    raise

    print(f"\n🎉 Vectorstore created with {collection.count()} chunks")

In [19]:
create_embeddings(chunks)

✅ Batch 1 added (100 chunks)
⚠️ Rate limit reached. Waiting 25 seconds...
⚠️ Rate limit reached. Waiting 25 seconds...
✅ Batch 2 added (100 chunks)
⚠️ Rate limit reached. Waiting 25 seconds...
✅ Batch 3 added (100 chunks)
⚠️ Rate limit reached. Waiting 25 seconds...
⚠️ Rate limit reached. Waiting 25 seconds...
✅ Batch 4 added (100 chunks)
⚠️ Rate limit reached. Waiting 25 seconds...
⚠️ Rate limit reached. Waiting 25 seconds...
✅ Batch 5 added (100 chunks)
⚠️ Rate limit reached. Waiting 25 seconds...
✅ Batch 6 added (77 chunks)

🎉 Vectorstore created with 577 chunks


# Nothing more to do here... right?

Wait! Didja think I'd forget??

In [20]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [21]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [43]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

## And now - let's build an Advanced RAG!

We will use these techniques:

1. Reranking - reorder the rank results
2. Query re-writing

In [23]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [24]:

def rerank(question, chunks):

    system_prompt = """
You are a document re-ranker.

You are provided with a user question and a list of retrieved document chunks.

Return ONLY valid JSON.

The JSON schema is:

{
    "order": [1,2,3,...]
}

The list must contain every chunk id exactly once, ordered from most relevant to least relevant.
"""

    user_prompt = f"""
Question:
{question}

Here are the chunks:

"""

    for index, chunk in enumerate(chunks):
        user_prompt += f"""
CHUNK ID: {index+1}

{chunk.page_content}

"""

    response = client1.models.generate_content(
        model=MODEL,
        contents=[
            system_prompt,
            user_prompt
        ],
        config={
            "response_mime_type": "application/json",
            "response_schema": RankOrder,
        }
    )

    if hasattr(response, "parsed") and response.parsed is not None:
        order = response.parsed.order
    else:
        order = RankOrder.model_validate_json(response.text).order

    print(order)

    return [chunks[i-1] for i in order]

In [25]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):

    # Create query embedding
    query_response = client1.models.embed_content(
        model=embedding_model,
        contents=question
    )

    query = query_response.embeddings[0].values

    # Search ChromaDB
    results = collection.query(
        query_embeddings=[query],
        n_results=RETRIEVAL_K
    )

    chunks = []

    for doc, meta in zip(
        results["documents"][0],
        results["metadatas"][0]
    ):
        chunks.append(
            Result(
                page_content=doc,
                metadata=meta
            )
        )

    return chunks

In [26]:
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)

In [29]:
for chunk in chunks:
    print(chunk.page_content[:15]+"...")

Senior Data Eng...
Additional HR N...
Career Progress...
HR Notes: Train...
Career Progress...
Marketing Inter...
Other HR Notes ...
Additional HR N...
Career Progress...
Performance His...


In [30]:
reranked = rerank(question, chunks)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


In [31]:
for chunk in reranked:
    print(chunk.page_content[:15]+"...")

Senior Data Eng...
Additional HR N...
Career Progress...
HR Notes: Train...
Career Progress...
Marketing Inter...
Other HR Notes ...
Additional HR N...
Career Progress...
Performance His...


In [32]:
question = "Who went to Manchester University?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

0


In [33]:
reranked = rerank(question, chunks)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


In [34]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)

0


In [35]:
reranked[0].page_content

"Skills, Education, and HR Notes\n\nJessica Liu's educational background, technical skills, professional development, work style, and HR feedback.\n\n## Other HR Notes\n- **Education:** BS in Computer Science from University of Manchester\n- **Skills:** Proficient in React, TypeScript, HTML/CSS, Jest for testing. Learning Next.js and GraphQL.\n- **Professional Development:** Completed Advanced React Patterns course (2023). Actively contributes to open-source projects.\n- **Work Style:** Prefers remote work. Strong written communicator. Participates actively in team standups and planning sessions.\n- **Feedback:** Reliable developer with good attention to UI details. Improving technical decision-making skills. Would benefit from more ownership of larger features."

In [36]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [44]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [67]:
def make_rag_messages(question, history, chunks):
    context = "\n\n".join(
        f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}"
        for chunk in chunks
    )

    system_prompt = SYSTEM_PROMPT.format(context=context)

    prompt = f"""{system_prompt}

Question:
{question}
"""

    return [
        {
            "role": "user",
            "parts": [
                {
                    "text": prompt
                }
            ]
        }
    ]

In [68]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = client1.models.generate_content(
        model=MODEL,
        contents=message,
    )
    return response.choices[0].message.content

In [69]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""

    response = client1.models.generate_content(
        model=MODEL,
        contents=message,
    )

    
    return response.text.strip()

In [70]:
rewrite_query("Who won the IIOTY award?", [])

'Who won the IIOTY award?'

In [ ]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [71]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)

    response = client1.models.generate_content(
        model=MODEL,
        contents=messages
    )

    return response.text, chunks

In [72]:
answer_question("Who won the IIOTY award?", [])

Who won the IIOTY award?
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


('Maxine Thompson won the Insurellm IIOTY (Innovator of the Year) award in 2023.',
 [Result(page_content="Senior Data Engineer and Achievements\n\nDetails Maxine's promotion to Senior Data Engineer, her leadership in data projects, and her 2023 IIOTY award.\n\n- **November 2018 - December 2020**: **Data Engineer**  \n  * In her new role, Maxine expanded her responsibilities to include designing comprehensive data models and improving data quality measures. Though she excelled in technical skills, communication issues with non-technical teams led to some project delays.  \n- **January 2021 - Present**: **Senior Data Engineer**  \n  * Maxine was promoted to Senior Data Engineer after successfully leading a pivotal project that improved data retrieval times by 30%. She now mentors junior engineers and is involved in strategic data initiatives, solidifying her position as a valued asset at Insurellm. She was recognized as Insurellm Innovator of the year in 2023, receiving the prestigious I

In [73]:
answer_question("Who went to Manchester University?", [])

Who attended Manchester University?
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


('Jessica Liu attended the University of Manchester, where she earned her BS in Computer Science.',
 [Result(page_content="Skills, Education, and HR Notes\n\nJessica Liu's educational background, technical skills, professional development, work style, and HR feedback.\n\n## Other HR Notes\n- **Education:** BS in Computer Science from University of Manchester\n- **Skills:** Proficient in React, TypeScript, HTML/CSS, Jest for testing. Learning Next.js and GraphQL.\n- **Professional Development:** Completed Advanced React Patterns course (2023). Actively contributes to open-source projects.\n- **Work Style:** Prefers remote work. Strong written communicator. Participates actively in team standups and planning sessions.\n- **Feedback:** Reliable developer with good attention to UI details. Improving technical decision-making skills. Would benefit from more ownership of larger features.", metadata={'type': 'employees', 'source': 'knowledge-base/employees/Jessica Liu.md'}),
  Result(page_con

In [74]:
answer_question("Who is avery?", [])

Who is Avery?
[1, 3, 15, 4, 9, 6, 2, 5, 7, 8, 10, 11, 12, 13, 14, 16, 17, 18, 19, 20]


("Hello! I'd be happy to introduce you to Avery Lancaster.\n\nAvery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. She founded the company in 2015 to innovate within the insurance tech sector, with Markellm serving as the company's inaugural product. \n\nUnder her leadership, Insurellm has grown significantly, expanding its portfolio to include Carllm, Homellm, and Rellm. Based in San Francisco, California, Avery has a background in product management and business analysis, having previously worked at Innovate Insurance Solutions and Edge Analytics before launching Insurellm. \n\nIn addition to her role as CEO, Avery is recognized for her focus on insurance innovation, risk management, and her commitment to fostering diversity and financial literacy within the community.",
 [Result(page_content='Avery Lancaster Profile\n\nOverview of Avery Lancaster including personal details, job title, location, and current salary as CEO of Insurellm.\n\n# Avery Lancaster